In [1]:
%cd ../
%load_ext autoreload
%autoreload 2

/home/christian/bachelor-project


In [2]:
# BiLSTM training for 12-hour wind vector forecasting

# This notebook trains `BiLSTMModel` on 10-min measurements to forecast the next 12 hours (72 steps)
# of u/v vector components per station.

import os
from omegaconf import OmegaConf
from loguru import logger
import pandas as pd

from src.database.database_service import DatabaseService
from src.weather_stations.weather_station_service import WeatherStationService
from src.measurements.measurement_service import MeasurementService
from src.model.variant.bilstm_model import BiLSTMModel


/home/christian/bachelor-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Load config and initialize services
cfg = OmegaConf.load("conf/config.yaml")

db = DatabaseService(cfg)
ws_service = WeatherStationService(cfg, db)
weather_stations = ws_service.load_from_database(only_relevant=True)

ms_service = MeasurementService(cfg, db, weather_stations)

logger.info(f"Loaded {len(weather_stations)} weather stations")


2025-09-12 14:39:36.826 | INFO     | src.weather_stations.weather_station_data_provider:load_from_database:228 - Loaded 50 weather stations from database
2025-09-12 14:39:36.839 | INFO     | __main__:<module>:10 - Loaded 50 weather stations


In [4]:
# Load measurements from DB
measurements_df = ms_service.load_all_measurements_from_database()
logger.info(f"Measurements: {len(measurements_df)} rows")

# Optional: downsample or filter timeframe for quicker training
start_date = pd.Timestamp('2023-01-01')
end_date = pd.Timestamp('2024-12-31')
train_df = measurements_df[(measurements_df['record_date']>=start_date) & (measurements_df['record_date']<=end_date)].copy()

test_df = measurements_df[measurements_df['record_date']>end_date].copy()

train_df.head()


2025-09-12 14:39:54.795 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 1000000)
2025-09-12 14:40:12.841 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 2000000)
2025-09-12 14:40:29.283 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 3000000)
2025-09-12 14:40:45.993 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 4000000)
2025-09-12 14:41:02.758 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loaded chunk of 1000000 rows (total so far: 5000000)
2025-09-12 14:41:22.460 | INFO     | src.measurements.measurement_data_provider:load_all_measurements_from_database:265 - Loa

,air_pressure,air_temperature_2m,air_temperature_5cm,average_wind_direction,average_wind_speed,dew_point_temperature,id,precipitation_duration,precipitation_indicator,quality_level,record_date,relative_humidity,station_id,sum_precipitation_height
0,1002.8,8.7,8.6,70.0,1.8,8.1,250001,10.0,1,3,2024-10-02 02:40:00,95.7,96,0.11
1,1002.7,8.7,8.6,60.0,2.0,8.0,250002,10.0,1,3,2024-10-02 02:50:00,95.6,96,0.06
2,1002.7,8.7,8.6,60.0,1.9,8.1,250003,10.0,1,3,2024-10-02 03:00:00,96.1,96,0.06
3,1002.8,8.7,8.6,70.0,1.9,8.1,250004,10.0,1,3,2024-10-02 03:10:00,96.1,96,0.12
4,1002.8,8.7,8.6,60.0,1.9,8.1,250005,10.0,1,3,2024-10-02 03:20:00,96.0,96,0.02


In [5]:
# Instantiate BiLSTM model
model = BiLSTMModel(
    history_steps=72,   # 12 hours of history
    horizon_steps=72,   # 12 hours forecast
    station_embedding_dim=8,
    hidden_size=128,
    num_layers=2,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    num_epochs=5,   # increase for real training
)
model


In [6]:
# Train model
model.train(train_df)


/home/christian/bachelor-project/src/model/variant/bilstm_model.py:428: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("station_id", sort=False, group_keys=False).apply(
2025-09-12 14:43:47.796 | INFO     | src.model.variant.bilstm_model:train:174 - Preparing training: sequences=4965215, stations=48, feature_dim=10, device=cuda
2025-09-12 14:43:47.932 | INFO     | src.model.variant.bilstm_model:train:190 - Model initialized: hidden_size=128, num_layers=2, dropout=0.1, params=650,000
2025-09-12 14:43:47.932 | INFO     | src.model.variant.bilstm_model:train:205 - DataLoader ready: samples=4965215, batch_size=64, batches_per_epoch=77582
2025-09-12 14:43:47.933 | INFO     | src.model.varian

In [7]:
# Save model
save_dir = "models/bilstm"
os.makedirs(save_dir, exist_ok=True)
model.save(save_dir)
logger.info(f"Saved BiLSTM model to {save_dir}")


2025-09-12 15:04:44.125 | INFO     | __main__:<module>:5 - Saved BiLSTM model to models/bilstm


In [17]:
new_model = BiLSTMModel(
    history_steps=72,   # 12 hours of history
    horizon_steps=72,   # 12 hours forecast
    station_embedding_dim=8,
    hidden_size=128,
    num_layers=2,
    dropout=0.1,
    batch_size=64,
    learning_rate=1e-3,
    num_epochs=5,   # increase for real training
)

new_model.load(save_dir)

In [18]:
# Quick prediction demo for a subset (uses last 72 steps per station)
preds = new_model.predict(measurements_df)
preds.head()


/home/christian/bachelor-project/src/model/variant/bilstm_model.py:428: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("station_id", sort=False, group_keys=False).apply(


,station_id,record_date,u_pred,v_pred
0,96,2025-09-10 08:30:00,-0.483345,-0.562206
1,96,2025-09-10 08:40:00,-0.478859,-0.562057
2,96,2025-09-10 08:50:00,-0.481697,-0.563381
3,96,2025-09-10 09:00:00,-0.486183,-0.564758
4,96,2025-09-10 09:10:00,-0.493049,-0.566627


In [19]:
evaluation = model.evaluate(test_df)


/home/christian/bachelor-project/src/model/variant/bilstm_model.py:428: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("station_id", sort=False, group_keys=False).apply(
2025-09-12 15:09:56.351 | INFO     | src.model.variant.bilstm_model:evaluate:13 - Evaluation: samples=1740060, batch_size=64, batches=27189
2025-09-12 15:10:42.805 | INFO     | src.model.variant.bilstm_model:evaluate:62 - Evaluation metrics: mae_u=0.0917, rmse_u=0.2234, mae_v=0.0474, rmse_v=0.2225, mae_speed=0.0547, rmse_speed=0.1404, mae_direction_deg=5.10°


In [20]:
per_h = model.evaluate_per_horizon(test_df, save_dir="artifacts/metrics")

2025-09-12 15:17:19.463 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:71 - Computed per-horizon metrics for 1..%d steps
2025-09-12 15:17:19.629 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:103 - Saved per-horizon plots to /home/christian/bachelor-project/artifacts/metrics


In [21]:
per_h = model.evaluate_per_horizon(test_df)

2025-09-12 15:19:12.368 | INFO     | src.model.variant.bilstm_model:evaluate_per_horizon:71 - Computed per-horizon metrics for 1..%d steps
